<a href="https://colab.research.google.com/github/LuanGibin/COMPARADOR-DE-COBERTURA-JORNALISTICA_PLN/blob/main/PLN_PROJETO_PR%C3%81TICO_EQUIPE_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem]


O **PROJETO PRÁTICO** deve ser feito utilizando o **Google Colab** com uma conta sua vinculada ao Gmail. O link do seu notebook armazenado no Google Drive e o link de um repositório no GitHub devem ser enviados usando o seguinte formulário:

> https://forms.gle/D4gLqP1iGgyn2hbH8


**IMPORTANTE**: A submissão deve ser feita até o dia **04/12 (sexta-feira)** APENAS POR UM INTEGRANTE DA EQUIPE, até às 23h59. Por favor, lembre-se de dar permissão de ACESSO IRRESTRITO para o professor da disciplina.

### **EQUIPE**

---

**POR FAVOR, PREENCHER OS INTEGRANDES DA SUA EQUIPE:**


**Integrante 01:**

`LUAN GIBIN FERNANDES PEREIRA | 11202230995`

**Integrante 02:**

`GABRIEL DE OLIVEIRA DALPIAN | 11202230264`

**Integrante 03:**

`CAROLINA F. RICCOMI ALVES | 11201921825`

**Integrante 04:**

`Por favor, informe o seu nome completo e RA:`

### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

Cada equipe deve selecionar um Grande Modelo de Linguagem (*Large Language Model - LMM*).



Por favor, informe os dados do LLM selecionado:

>


**LLM**: gemini-3.8-flash

>

**Link para a documentação oficial**: https://aistudio.google.com/docs/models/gemini-3.8-flash



### **API (Opcional)**
---

O Projeto não irá utilizar API.
A obtenção dos dados será feita por meio da biblioteca open source **feedparser**

**Link para a documentação oficial do feedparser v6.0.14:** https://feedparser.readthedocs.io/en/stable/index.html




### **DESCRIÇÃO**
---

Implementar um `notebook` no `Google Colab` que faça uso do framework **`LangChain`** (obrigatório) e de um **LLM** aplicando, no mínimo, DUAS técnicas de PLN. As técnicas podem ser aplicada em qualquer córpus obtido a partir de uma **API** ou a partir de uma página Web.

O **LLM** e a **API** selecionados devem ser informados na seguinte planilha:

> https://docs.google.com/spreadsheets/d/1iIUZcwnywO7RuF6VEJ8Rx9NDT1cwteyvsnkhYr0NWtU/edit?usp=sharing

>
As seguintes técnicas de PLN podem ser usadas:

*   Correção Gramatical
*   Classificação de Textos
*   Análise de Sentimentos
*   Detecção de Emoções
*   Extração de Palavras-chave
*   Tradução de Textos
*   Sumarização de Textos
*   Similaridade de Textos
*   Reconhecimento de Entidades Nomeadas
*   Sistemas de Perguntas e Respostas
>

**IMPORTANTE:** É obrigatório usar o e-mail da UFABC.


### **CRITÉRIOS DE AVALIAÇÃO**
---


Serão considerados como critérios de avaliação os seguintes pontos:

* Uso do framework **`LangChain`**.

* Escolha e uso de um **LLM**.

* Escolha e uso de uma **API** ou **Página Web**.

* Projeto disponível no Github.

* Apresentação (10 a 15 minutos).

* Criatividade no uso do framework **`LangChain`** em conjunto com o **LLM** e a **API**.



Também é interessante explorar técnicas como:
*   Geração de Dados Sintéticos
*   Geração Aumentada por Recuperação (RAG)
*   Agentes


**IMPORTANTE**: todo o código do notebook deve ser documentado e EXECUTADO. Código sem execução não será considerado.

### **IMPLEMENTAÇÃO**
---

**1.** Instalação da biblioteca feedparser e exibição da versão instalada

In [ ]:
# Instalação da biblioteca feedparser
!pip install feedparser

In [ ]:
# Import da biblioteca e exibição da versão instalada

import feedparser

print(f"Versão atual do feedparser: {feedparser.__version__}")

Versão atual do feedparser: 6.0.14


**2.** Leitura do RSS do G1 (https://g1.globo.com/rss/g1/) e RSS (https://feeds.bbci.co.uk/portuguese/rss.xml) do BBC Brasil por meio do feedparser

In [10]:
"""Módulo de Coleta e Normalização de Feeds RSS para PLN.

Este script realiza a extração de notícias dos feeds RSS do G1 e da BBC Brasil
utilizando a biblioteca `feedparser` (v6.0.14). Os dados extraídos passam por
higienização textual (remoção de HTML) e normalização temporal (conversão de
UTC para o fuso horário de Brasília) para estruturar um corpus limpo e
padronizado, pronto para tarefas de Processamento de Linguagem Natural (PLN).
"""

from calendar import timegm
from collections import Counter
from datetime import datetime, timezone
import re
from typing import Any
from zoneinfo import ZoneInfo

import feedparser

# Dicionário mapeando os identificadores dos veículos às suas respectivas URLs RSS
FEEDS_JORNALISTICOS: dict[str, str] = {
    "G1": "https://g1.globo.com/rss/g1/",
    "BBC_Brasil": "https://feeds.bbci.co.uk/portuguese/rss.xml",
}

# Fuso horário alvo para alinhamento temporal das publicações (UTC-3)
FUSO_HORARIO_PADRAO = ZoneInfo("America/Sao_Paulo")


def limpar_html(texto_bruto: str) -> str:
  """Remove marcações HTML e normaliza espaços em branco de uma string.

  Feeds RSS frequentemente embutem tags HTML (como `<img />`, `<br />` ou
  `<p>`) dentro das tags de descrição (`<description>` / `summary`). Esta
  função remove essas marcações para evitar ruído na tokenização de PLN.

  Args:
      texto_bruto: String original extraída do feed contendo possíveis tags HTML.

  Returns:
      String sanitizada, contendo apenas texto puro e espaçamento simples.
  """
  if not texto_bruto:
    return ""

  # Substitui qualquer tag HTML por um espaço para evitar aglutinação de palavras
  sem_tags = re.sub(r"<.*?>", " ", texto_bruto)

  # Colapsa múltiplos espaços, tabulações ou quebras de linha em um único espaço
  return re.sub(r"\s+", " ", sem_tags).strip()


def normalizar_data_publicacao(
    entry: feedparser.FeedParserDict, fuso_destino: ZoneInfo
) -> str | None:
  """Converte a data de publicação do feed (UTC) para o fuso horário local.

  O `feedparser` converte automaticamente qualquer string de data padrão
  (RFC 822 / ISO 8601) para uma `struct_time` em UTC no atributo
  `published_parsed`. Esta função interpreta essa tupla corretamente em UTC
  e a converte para o fuso horário especificado no formato ISO 8601.

  Args:
      entry: Objeto de entrada individual retornado pelo `feedparser`.
      fuso_destino: Objeto `ZoneInfo` representando o fuso horário desejado.

  Returns:
      String formatada no padrão ISO 8601 com offset de fuso horário
      (ex: '2026-11-17T20:30:24-03:00') ou `None` se a data não existir.
  """
  if not hasattr(entry, "published_parsed") or not entry.published_parsed:
    return None

  # `timegm` interpreta a struct_time estritamente como UTC (diferente de `mktime`)
  timestamp_utc = timegm(entry.published_parsed)
  dt_utc = datetime.fromtimestamp(timestamp_utc, tz=timezone.utc)

  # Converte o objeto ciente de UTC para o fuso horário de destino
  dt_local = dt_utc.astimezone(fuso_destino)
  return dt_local.isoformat()


def coletar_noticias(fontes: dict[str, str]) -> list[dict[str, Any]]:
  """Coleta, higieniza e estrutura notícias de múltiplos feeds RSS.

  Args:
      fontes: Dicionário no formato `{"Nome_Veiculo": "URL_do_RSS"}`.

  Returns:
      Lista de dicionários padronizados, onde cada dicionário representa uma
      notícia com metadados e campos textuais prontos para vetorização/PLN.
  """
  dataset_noticias: list[dict[str, Any]] = []

  # Identificação explícita na requisição HTTP para evitar bloqueios (HTTP 403)
  user_agent = "Mozilla/5.0 (ProjetoPLN-ComparadorNoticias/1.0)"

  for nome_veiculo, url in fontes.items():
    print(f"[INFO] Lendo feed RSS: {nome_veiculo}...")
    feed = feedparser.parse(url, agent=user_agent)

    # Verifica o 'bozo bit' do feedparser, que sinaliza XML malformado ou falha de rede
    if feed.bozo:
      print(
          f"  [AVISO] Anomalia detectada ao processar {nome_veiculo}: "
          f"{feed.bozo_exception}"
      )

    for entry in feed.entries:
      titulo = entry.get("title", "").strip()

      # Alguns feeds usam 'summary' e outros 'description'; o fallback garante compatibilidade
      resumo_bruto = entry.get("summary", entry.get("description", ""))
      resumo_limpo = limpar_html(resumo_bruto)

      # Extrai categorias/editorias caso o portal as disponibilize no XML
      tags = [
          tag.term for tag in entry.get("tags", []) if hasattr(tag, "term")
      ]

      noticia_estruturada: dict[str, Any] = {
          "veiculo": nome_veiculo,
          "titulo": titulo,
          "resumo": resumo_limpo,
          # Concatenação recomendada para maximizar o contexto semântico em embeddings/TF-IDF
          "texto_completo_pln": f"{titulo}. {resumo_limpo}".strip(),
          "data_publicacao": normalizar_data_publicacao(
              entry, FUSO_HORARIO_PADRAO
          ),
          "tags": tags,
          "url": entry.get("link", ""),
      }
      dataset_noticias.append(noticia_estruturada)

  return dataset_noticias


def exibir_relatorio_coleta(
    noticias: list[dict[str, Any]], fontes: dict[str, str]
) -> None:
  """Exibe métricas quantitativas da coleta e uma amostra de cada veículo.

  Args:
      noticias: Lista de notícias estruturadas retornada por
        `coletar_noticias`.
      fontes: Dicionário original de fontes para iterar sobre os veículos.
  """
  contagem_por_veiculo = Counter(n["veiculo"] for n in noticias)

  print("\n" + "=" * 65)
  print(f"RESUMO DA COLETA | Total Geral: {len(noticias)} notícias")
  print("=" * 65)

  for veiculo in fontes.keys():
    qtd = contagem_por_veiculo.get(veiculo, 0)
    print(f"  • {veiculo:<12}: {qtd} notícias extraídas")

  print("-" * 65)
  print("AMOSTRA DE VALIDAÇÃO (1ª notícia de cada veículo):")
  print("-" * 65)

  for veiculo in fontes.keys():
    amostra = next((n for n in noticias if n["veiculo"] == veiculo), None)
    if amostra:
      print(f"VEÍCULO : {amostra['veiculo']}")
      print(f"DATA    : {amostra['data_publicacao']}")
      print(f"TÍTULO  : {amostra['titulo']}")
      print(f"RESUMO  : {amostra['resumo'][:160]}...")
      print("-" * 65)


if __name__ == "__main__":
  # Execução principal do pipeline de extração
  corpus_noticias = coletar_noticias(FEEDS_JORNALISTICOS)
  exibir_relatorio_coleta(corpus_noticias, FEEDS_JORNALISTICOS)

[INFO] Lendo feed RSS: G1...
[INFO] Lendo feed RSS: BBC_Brasil...

RESUMO DA COLETA | Total Geral: 131 notícias
  • G1          : 100 notícias extraídas
  • BBC_Brasil  : 31 notícias extraídas
-----------------------------------------------------------------
AMOSTRA DE VALIDAÇÃO (1ª notícia de cada veículo):
-----------------------------------------------------------------
VEÍCULO : G1
DATA    : 2026-11-17T20:53:08-03:00
TÍTULO  : Suspeitos de matar mulher por engano são apreendidos no PI; alvo seria companheira de membro de facção
RESUMO  : Morte em José de Freitas: mulher foi morta a tiros por engano ontem à noite Um homem foi preso e quatro adolescentes foram apreendidos suspeitos de envolvimento...
-----------------------------------------------------------------
VEÍCULO : BBC_Brasil
DATA    : 2026-11-17T17:07:30-03:00
TÍTULO  : O 'exército das avós': os números que mostram as responsáveis pelo pico de comparecimento nas eleições
RESUMO  : Dados do TSE confirmam agora que os vídeos